In [ ]:
import json

set_of_questions = []
with open('set_of_question.json', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                set_of_questions.append(json.loads(line))
            except Exception as e:
                pass  # or handle error as desired

In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-3-4b-it",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/user01/miniconda3/envs/tetris_finetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


We now add LoRA adapters for parameter efficient fine-tuning, allowing us to train only 1% of all model parameters efficiently.

**[NEW]** We also support fine-tuning only the vision component, only the language component, or both. Additionally, you can choose to fine-tune the attention modules, the MLP layers, or both!

In [5]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


In [ ]:
import json

redis_docs_path = "/home/user01/rahnema/cache/redis_docs.jsonl"
json_data = []
with open(redis_docs_path, 'r', encoding='utf-8') as f:
    for line in f:
        json_data.append(json.loads(line))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.9/339.9 kB 7.0 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import list_repo_files

repo_id = "alizali/torob_images_parquet_resize"
all_files = list_repo_files(repo_id, repo_type="dataset")
parquet_files = [f for f in all_files if f.endswith('.parquet')]


In [ ]:
len(parquet_files)

258

In [ ]:
from datasets import load_dataset
import numpy as np
from PIL import Image
# import matplotlib.pyplot as plt
import os

img_dataset = load_dataset(
    "/home/user01/datasets/torob_images_parquet",
    data_files=parquet_files)
img_dataset = img_dataset.shuffle(seed=42)

[1, 3, 142]


1_entities_dataset_v2.parquet:   0%|          | 0.00/879M [00:00<?, ?B/s]

3_entities_dataset_v2.parquet:   0%|          | 0.00/787M [00:00<?, ?B/s]

142_entities_dataset_v2.parquet:   0%|          | 0.00/367M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
import json
import os
import numpy as np
from PIL import Image
from datasets import load_dataset
import io


json_lookup = {}
for item in json_data:
    image_key = item.get('image') or item.get('image_url')
    if image_key:
        image_id = image_key.split('/')[-1]
        json_lookup[image_id] = item

def add_json_info(example):
    image_id = example.get('filename')
    example['entity_name'] = []
    if image_id and image_id in json_lookup:
        info = json_lookup[image_id]
        prompt = '''سوالات زیر را به ترتیب در هر خط جواب بده. به طور مثال اگر تصویر یک تی‌شرت قرمز را فرستادم و پرسیدم:
        این تصویر چه محصولی است؟ چه رنگی دارد؟
        جواب تو این خواهد بود:
        1. تی‌شرت
        2. قرمز
        جواب تو فقط همان دو خط بالا خواهد بود و هیج کلمه‌ی اضافه‌ای نخواهی گفت. مثلا برای سوال اول نمی‌گویی این یک تیشرت است. یا قبل از آن که جواب را بدهی نمی‌گویی باشد سوالات‌ات را بپرس. یا نمی‌گویی فهمیدم.
        '''
        prompt += 'این تصویر چه محصولی است؟ '
        ent_num = 1
        if isinstance(info.get('product', []), list):
            ans = ''.join(' ' + f for f in info.get('product', []) if f != 'بی‌ربط')
            answers = ''.join(f'{ent_num}. ' + ans)
        else:
            answers = ''.join(f'{ent_num}. ' + info.get('product', []))
        example['entity_name'].append('product')
        for k in info.keys():
            if info[k] != None:
                if len(info[k]) > 0 and k != 'product' and k in set_of_questions.keys():
                    ent_num += 1
                    if isinstance(info[k], list):
                        ans = ''.join(' ' + f for f in info[k] if f != 'بی‌ربط')
                        ans = ''.join(f'{ent_num}. ' + ans)
                    elif info[k] != 'بی‌ربط':
                        ans = ''.join(f'{ent_num}. ' + info[k])
                    else:
                        continue
                    example['entity_name'].append(k)
                    prompt += set_of_questions[k]
                    answers += '\n' + ans
        example['question'] = prompt
        example['tags'] = answers
    else:
        example['question'] = ''
        example['tags'] = ''
    return example

# Apply the initial combination
combined_dataset = img_dataset['train'].map(add_json_info)


Map:   0%|          | 0/25058 [00:00<?, ? examples/s]

In [11]:
from tqdm import tqdm

def format_for_pipeline(example):
    """
    Transforms a single example into the target multi-modal conversation format,
    ensuring the image type is PIL.PngImagePlugin.PngImageFile.
    """
    img_array = np.frombuffer(example["array"], dtype=np.uint8)

    question = example.get('question', '')
    answer = example.get('tags', '')

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image", "image": img_array.tolist()},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": answer}
            ],
        },
    ]

    return {"messages": messages}


In [12]:
batch_size = 1000
output_file_path = 'final_dataset.jsonl'
current_batch = []

print("Generating new dataset file with corrected PNG format...")
with open(output_file_path, 'w', encoding='utf-8') as f:
    for sample in tqdm(combined_dataset, desc="Preparing data"):
        formatted_sample = format_for_pipeline(sample)
        current_batch.append(formatted_sample)
        if len(current_batch) == batch_size:
            for item in current_batch:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
            current_batch = []
    if current_batch:
        for item in current_batch:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"Dataset successfully saved to {output_file_path}")

Generating new dataset file with corrected PNG format...


Preparing data: 100%|██████████| 25058/25058 [11:54<00:00, 35.06it/s]


Dataset successfully saved to final_dataset.jsonl


In [13]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-3"
)

In [14]:
def load_and_prepare_chunk(file_handle, chunk_size):
    """
    Reads a chunk from the file, decodes the Base64 images, and loads
    them using Image.open, which correctly handles the PNG format.
    """
    chunk = []
    for _ in range(chunk_size):
        line = file_handle.readline()
        if not line: break

        data = json.loads(line.strip())
        image_content = data['messages'][0]['content'][1]

        if image_content:
            pil_image = Image.fromarray(np.array(image_content['image']).reshape((224,224,3)), 'RGB')

            with io.BytesIO() as buffer:
                pil_image.save(buffer, format='PNG')
                buffer.seek(0)
                png_image_file = Image.open(buffer)
                png_image_file.load()

            image_content['type'] = 'image'
            image_content['image'] = png_image_file

        chunk.append(data)
    return chunk


In [ ]:
CHUNK_SIZE = 500
NUM_EPOCHS = 1
FILE_PATH = 'final_dataset.jsonl'

print("Loading the initial chunk to initialize the trainer...")
file_iterator = open(FILE_PATH, 'r', encoding='utf-8')
initial_chunk = load_and_prepare_chunk(file_iterator, CHUNK_SIZE)

In [17]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig


if not initial_chunk:
    raise ValueError("Dataset file is empty or could not be read.")

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    train_dataset=initial_chunk,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        gradient_checkpointing = True,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
        save_strategy="steps",
        save_steps = 100,
        optim = "adamw_torch_fused",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )
)

Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
for epoch in range(NUM_EPOCHS):
    print(f"\n--- Starting Epoch {epoch + 1}/{NUM_EPOCHS} ---")
    chunk_num = 0

    if epoch > 0:
        file_iterator.close()
        file_iterator = open(FILE_PATH, 'r', encoding='utf-8')

    while True:
        chunk_num += 1
        print(f"\n[Epoch {epoch + 1}] Processing chunk {chunk_num}...")

        trainer.train()

        current_chunk = load_and_prepare_chunk(file_iterator, CHUNK_SIZE)
        if not current_chunk:
            print("Finished epoch.")
            break
        trainer.train_dataset = current_chunk


file_iterator.close()
print("\n--- Training Complete ---")


--- Starting Epoch 1/1 ---

[Epoch 1] Processing chunk 1...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 38,497,792 of 4,338,577,264 (0.89% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,3.487800
20,0.732700
30,0.432200
40,0.286200
50,0.324300
60,0.287400



[Epoch 1] Processing chunk 2...


/tmp/ipython-input-1146022991.py:16: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil_image = Image.fromarray(np.array(image_content['image']).reshape((224,224,3)), 'RGB')
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 38,497,792 of 4,338,577,264 (0.89% trained)


Step,Training Loss
10,0.306500
20,0.331900
30,0.275000
40,0.257300
50,0.259800
60,0.266200



[Epoch 1] Processing chunk 3...


/tmp/ipython-input-1146022991.py:16: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil_image = Image.fromarray(np.array(image_content['image']).reshape((224,224,3)), 'RGB')
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 38,497,792 of 4,338,577,264 (0.89% trained)


Step,Training Loss
10,0.276700


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can modify the instruction and input—just leave the output blank.

We'll use the best hyperparameters for inference on Gemma: `top_p=0.95`, `top_k=64`, and `temperature=1.0`.

In [ ]:
FastVisionModel.for_inference(model)  # Enable for inference!

image = dataset[10]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt=True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache=True, temperature = 1.0, top_p = 0.95, top_k = 64)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, use Hugging Face’s `push_to_hub` for online saving, or `save_pretrained` for local storage.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
processor.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# processor.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastVisionModel

    model, processor = FastVisionModel.from_pretrained(
        model_name="lora_model",  # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit=True,  # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model)  # Enable for inference!

FastVisionModel.for_inference(model)  # Enable for inference!

sample = dataset[1]
image = sample["image"].convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": sample["text"],
            },
            {
                "type": "image",
            },
        ],
    },
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor.tokenizer, skip_prompt=True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache=True, temperature = 1.0, top_p = 0.95, top_k = 64)

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", processor,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", processor, token = "PUT_HERE")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>


In [1]:
dependencies = """unsloth
torch
re
os
transformers==4.56.2
peft
trl==0.22.2
bitsandbytes
accelerate
xformers==0.0.32.post2 # or 0.0.29.post3 depending on torch version
triton
cut_cross_entropy
unsloth_zoo
sentencepiece
protobuf
datasets>=3.4.1,<4.0.0
huggingface_hub>=0.34.0
hf_transfer
redis==7.0.1
json
typing
numpy
Pillow
matplotlib
tqdm
io
"""

with open('requirements.txt', 'w') as f:
    f.write(dependencies)

print("Dependencies saved to requirements.txt")

Dependencies saved to requirements.txt


In [2]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
